In [ ]:
import pandas as pd
import plotnine as gg
from essential.data import load_fitness_data
import scanpy as sc

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

In [ ]:
fitness_df = load_fitness_data()

## Interpretation

### Nucleotide toxicity

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"] = "none"

In [ ]:
nuc_genes = ["dut", "dcd", "thyA", "tmk"]

colors = {
    "dcd": "#EE67EC",
    "dut": "#EE8045",
    "thyA": "#00B9E3",
    "tmk": "#93AA00",
}


fitness_df_subset = (
    fitness_df.loc[lambda x: x["gene"].isin(nuc_genes)]
    .groupby("gene")[["T1", "T2", "T3", "T4"]]
    .mean()
    .stack()
    .to_frame("fitness")
    .reset_index()
    .rename(columns={"level_1": "time"})
    .assign(time=lambda x: x["time"].replace({"T1": "3H", "T2": "6H", "T3": "24H", "T4": "24+6H"}))
    .assign(
        time=lambda x: pd.Categorical(
            x["time"], categories=["3H", "6H", "24H", "24+6H"], ordered=True
        )
    )
)
fig = (
    gg.ggplot(fitness_df_subset, gg.aes(x="time", y="fitness", color="gene", group="gene"))
    + gg.geom_line()
    + gg.geom_point()
    + gg.scale_color_manual(values=colors)
    + gg.theme_minimal()
    + gg.labs(x="time", y="fitness", color="")
    + gg.theme(
        axis_text=gg.element_text(size=6),
        axis_title=gg.element_text(size=7),
        legend_text=gg.element_text(size=6),
        legend_key_size=0.4,
        figure_size=(3, 2),
    )
)
fig.save("figures/nucleotide_toxicity.svg")
fig

In [ ]:
adata_subset = adata[adata.obs["gene"].isin(nuc_genes)]
adata_subset.obs["gene"] = adata_subset.obs["gene"].astype(str)
adata_subset.uns["gene_colors"] = [
    colors[gene] for gene in sorted(adata_subset.obs["gene"].unique())
]
sc.pl.heatmap(
    adata_subset,
    groupby="gene",
    var_names=["recA", "lexA"],
    swap_axes=False,
    figsize=(1, 4),
    save="recA_lexA_heatmap.svg",
)

In [ ]:
adata_case_subset = adata_case[adata_case.obs["gene"].isin(colors)]
adata_case_subset.obs["gene"] = adata_case_subset.obs["gene"].astype(str)


fig = (
    gg.ggplot(
        adata_case.obs,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
    )
    + gg.geom_point()
    + gg.geom_point(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene"),
        data=adata_case_subset.obs,
    )
    + gg.theme_minimal()
    + gg.scale_color_manual(values=colors)
    + gg.labs(x="UMAP1", y="UMAP2", color="")
)
fig.save("figures/nucleotide_umap.png", dpi=300)
fig

### Coenzyme A

In [ ]:
adata_case

In [ ]:
genes = ["coaA", "dfp", "coaD", "coaE"]
adata_case_subset = adata_case[adata_case.obs["gene"].isin(genes)]
adata_case_subset.obs["gene"] = adata_case_subset.obs["gene"].astype(str)


fig = (
    gg.ggplot(
        adata_case.obs,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
    )
    + gg.geom_point()
    + gg.geom_point(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene"),
        data=adata_case_subset.obs,
    )
    + gg.theme_minimal()
    + gg.labs(x="UMAP1", y="UMAP2", color="")
)
fig.save("figures/coenzyme_a_umap.png", dpi=300)
fig

In [ ]:
adata_case_subset.obs["gene"].unique()

In [ ]:
genes = ["coaA", "dfp", "coaD", "coaE"]
genes = {
    "coaA": "#F76E65",
    "dfp": "#C372F8",
}


fitness_df_subset = (
    fitness_df.loc[lambda x: x["gene"].isin(genes)]
    .groupby("gene")[["T1", "T2", "T3", "T4"]]
    .mean()
    .stack()
    .to_frame("fitness")
    .reset_index()
    .rename(columns={"level_1": "time"})
    .assign(time=lambda x: x["time"].replace({"T1": "3H", "T2": "6H", "T3": "24H", "T4": "24+6H"}))
    .assign(
        time=lambda x: pd.Categorical(
            x["time"], categories=["3H", "6H", "24H", "24+6H"], ordered=True
        ),
    )
)
fig = (
    gg.ggplot(fitness_df_subset, gg.aes(x="time", y="fitness", color="gene", group="gene"))
    + gg.geom_line()
    + gg.geom_point()
    + gg.scale_color_manual(values=genes)
    + gg.theme_minimal()
    + gg.labs(x="time", y="fitness", color="")
    + gg.theme(
        axis_text=gg.element_text(size=6),
        axis_title=gg.element_text(size=7),
        legend_text=gg.element_text(size=6),
        legend_key_size=0.4,
        figure_size=(3, 2),
    )
)
# fig.save("figures/nucleotide_toxicity.svg")
fig

### Heme biosynthesis

In [ ]:
genes = ["hemL", "hemB", "hemC", "hemD", "hemE"]
adata_case_subset = adata_case[adata_case.obs["gene"].isin(genes)]
adata_case_subset.obs["gene"] = adata_case_subset.obs["gene"].astype(str)


fig = (
    gg.ggplot(
        adata_case.obs,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
    )
    + gg.geom_point()
    + gg.geom_point(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene"),
        data=adata_case_subset.obs,
    )
    + gg.theme_minimal()
    + gg.labs(x="UMAP1", y="UMAP2", color="")
)
fig.save("figures/heme_umap.png", dpi=300)
fig

In [ ]:
# oxydative_stress_response_genes = [
#     "katE",
#     "katG",
#     "ahpC",
#     "sodA",
#     "sodB",
#     "trxA",
#     "grxA",
#     # "gorA",
#     # "gorG",
#     "cfa",
#     "spy",
#     "cpxP",
#     "dps",
#     "ftnA",
#     "bfr",
#     # "suf",
# ]

In [ ]:
zinc_genes = [
    "zapA",
    "zapB",
    "zapC",
    "zapD",
    "zapE",
    "rpmE",
    "pyrC",
    "glyA",
    "zur",
]

# stress_genes = ["rpmE", "pyrC", "glyA", "zur"]

In [ ]:
adata_subset = adata_case[adata_case.obs["gene"].isin(genes)]
adata_subset.obs["gene"] = adata_subset.obs["gene"].astype(str)
# adata_subset.uns["gene_colors"] = [
#     colors[gene] for gene in sorted(adata_subset.obs["gene"].unique())
# ]
sc.pl.heatmap(
    adata_subset,
    groupby="gene",
    var_names=zinc_genes,
    swap_axes=False,
    figsize=(4, 4),
    save="recA_lexA_heatmap.svg",
)
sc.pl.dotplot(
    adata_subset,
    groupby="gene",
    var_names=zinc_genes,
    swap_axes=False,
    figsize=(4, 4),
    save="recA_lexA_dotplot.svg",
)

### kds

In [ ]:
genes = ["kdsD", "kdsA", "kdsC", "kdsB"]
adata_case_subset = adata_case[adata_case.obs["gene"].isin(genes)]
adata_case_subset.obs["gene"] = adata_case_subset.obs["gene"].astype(str)

fig = (
    gg.ggplot(
        adata_case.obs,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
    )
    + gg.geom_point()
    + gg.geom_point(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene"),
        data=adata_case_subset.obs,
    )
    + gg.theme_minimal()
    + gg.labs(x="UMAP1", y="UMAP2", color="")
)
# fig.save("figures/heme_umap.png", dpi=300)
fig

In [ ]:
spy, osmB, yebE

In [ ]:
stress_response_genes = [
    "spy",
    "osmB",
    "yebE",
    "wzc",
    "gmd",
]

In [ ]:
genes = ["kdsA", "kdsC", "kdsB"]
colors = {
    "kdsA": "#F17163",
    "kdsC": "#2CB4BC",
    "kdsB": "#6DAF08",
}


adata_subset = adata_case[adata_case.obs["gene"].isin(genes)]
adata_subset.obs["gene"] = adata_subset.obs["gene"].astype(str)
adata_subset.uns["gene_colors"] = [
    colors[gene] for gene in sorted(adata_subset.obs["gene"].unique())
]
sc.pl.heatmap(
    adata_subset,
    groupby="gene",
    var_names=stress_response_genes,
    swap_axes=False,
    figsize=(4, 4),
    save="kds_genes_heatmap.svg",
)

In [ ]:
colors = {
    "kdsA": "#F17163",
    "kdsC": "#2CB4BC",
    "kdsB": "#6DAF08",
}


fitness_df_subset = (
    fitness_df.loc[lambda x: x["gene"].isin(colors.keys())]
    .groupby("gene")[["T1", "T2", "T3", "T4"]]
    .mean()
    .stack()
    .to_frame("fitness")
    .reset_index()
    .rename(columns={"level_1": "time"})
    .assign(time=lambda x: x["time"].replace({"T1": "3H", "T2": "6H", "T3": "24H", "T4": "24+6H"}))
    .assign(
        time=lambda x: pd.Categorical(
            x["time"], categories=["3H", "6H", "24H", "24+6H"], ordered=True
        )
    )
)
fig = (
    gg.ggplot(fitness_df_subset, gg.aes(x="time", y="fitness", color="gene", group="gene"))
    + gg.geom_line()
    + gg.geom_point()
    + gg.scale_color_manual(values=colors)
    + gg.theme_minimal()
    + gg.labs(x="time", y="fitness", color="")
    + gg.theme(
        axis_text=gg.element_text(size=6),
        axis_title=gg.element_text(size=7),
        legend_text=gg.element_text(size=6),
        legend_key_size=0.4,
        figure_size=(3, 2),
    )
)
fig.save("figures/kds_toxicity.svg")
fig